In [ ]:
import importlib.util, json, math, random, re, subprocess, sys
from pathlib import Path
_REQUIRED = {"numpy": "numpy", "pandas": "pandas", "pyarrow": "pyarrow", "networkx": "networkx", "sklearn": "scikit-learn"}
_missing = [package for module, package in _REQUIRED.items() if importlib.util.find_spec(module) is None]
if _missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_missing])
import numpy as np
import pandas as pd
import networkx as nx
from sklearn.feature_extraction.text import TfidfVectorizer
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "config.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Could not find shared config.json from the notebook working directory")
CONFIG = json.loads((ROOT / "config.json").read_text(encoding="utf-8"))
SEED = int(CONFIG["random_seed"])
random.seed(SEED)
np.random.seed(SEED)
for key in ("data_dir", "artifact_dir", "results_dir"):
    (ROOT / CONFIG[key]).mkdir(parents=True, exist_ok=True)
ARTIFACTS = ROOT / CONFIG["artifact_dir"]
RESULTS = ROOT / CONFIG["results_dir"]
print(f"Setup complete; smoke_test_mode={CONFIG['smoke_test_mode']}; seed={SEED}")

# 04 — Retrieval, fusion, and reranking smoke path

The pipeline gates on known age/term eligibility before scoring, then min-max normalizes dense and BM25/ KG preference components, retrieves top-8, and fuses ranking scores. In this smoke run, TF-IDF and exact NumPy/FAISS fallbacks are used locally; cross-encoder scores are not probabilities. Alpha/beta and KG preference weights are not tuned because the only labels file is blank.

In [ ]:
policies_df = pd.read_parquet(ARTIFACTS / "policies.parquet")
queries_df = pd.read_csv(ARTIFACTS / "queries.csv")
intents = {row["query_id"]: row for row in json.loads((ARTIFACTS / "intents.json").read_text(encoding="utf-8"))}
kg = nx.read_graphml(ARTIFACTS / "kg.graphml")
finetune_status = json.loads((RESULTS / "finetune.json").read_text(encoding="utf-8"))
label_status = json.loads((RESULTS / "data_status.json").read_text(encoding="utf-8"))

policy_records = policies_df.to_dict(orient="records")
policy_by_id = {str(row["policy_id"]): row for row in policy_records}
policy_ids = sorted(policy_by_id)
documents = {
    pid: " ".join(str(policy_by_id[pid][key]) for key in ("policy_name", "category", "financial_goal", "coverage_type", "description"))
    for pid in policy_ids
}
vectorizer = TfidfVectorizer(ngram_range=(1, 2), norm="l2")
document_matrix = vectorizer.fit_transform([documents[pid] for pid in policy_ids])
try:
    import faiss
    HAS_FAISS = True
except (ImportError, OSError):
    HAS_FAISS = False

# Smoke-mode weights exercise the score path only; they are not selected weights.
SMOKE_ALPHA = 0.5
SMOKE_BETA = 0.5
SMOKE_KG_PREF_WEIGHTS = {"goal": 1 / 3, "risk": 1 / 3, "premium": 1 / 3}


def backend_plan(faiss_available, encoder_available, network_allowed, cross_encoder_loaded):
    return {
        "vector": "faiss_exact_ip" if faiss_available else "numpy_exact",
        "encoder": "pretrained_biencoder" if encoder_available and network_allowed else "tfidf_fallback",
        "cross_encoder": "cross_encoder_ranking_score" if cross_encoder_loaded else "dense_ranking_score_fallback",
        "network": "allowed" if network_allowed else "disabled_local_only",
    }


BACKENDS = backend_plan(HAS_FAISS, False, bool(CONFIG["allow_model_downloads"]), False)


def minmax(values):
    values = np.asarray(values, dtype=float)
    if values.size == 0:
        return values
    lo, hi = float(values.min()), float(values.max())
    return np.zeros_like(values) if math.isclose(lo, hi, abs_tol=1e-12) else (values - lo) / (hi - lo)


def tokens(text):
    return re.findall(r"[a-z0-9]+", str(text).lower())


def bm25_scores(query, ids):
    docs = [tokens(documents[pid]) for pid in ids]
    query_terms = tokens(query)
    n_docs = len(docs)
    if not n_docs:
        return np.array([], dtype=float)
    lengths = [len(doc) for doc in docs]
    avg_len = sum(lengths) / n_docs or 1.0
    doc_freq = {term: sum(term in set(doc) for doc in docs) for term in set(query_terms)}
    scores = []
    for doc, length in zip(docs, lengths):
        score = 0.0
        for term in query_terms:
            tf = doc.count(term)
            if not tf:
                continue
            idf = math.log(1 + (n_docs - doc_freq[term] + 0.5) / (doc_freq[term] + 0.5))
            score += idf * tf * 2.2 / (tf + 1.2 * (0.25 + 0.75 * length / avg_len))
        scores.append(score)
    return np.asarray(scores, dtype=float)


def dense_scores(query, ids, use_faiss=None):
    if not ids:
        return np.array([], dtype=float)
    if use_faiss is None:
        use_faiss = HAS_FAISS
    positions = [policy_ids.index(pid) for pid in ids]
    matrix = document_matrix[positions].toarray().astype("float32")
    query_vector = vectorizer.transform([query]).toarray().astype("float32")
    if use_faiss:
        index = faiss.IndexFlatIP(matrix.shape[1])
        index.add(matrix)
        distances, matches = index.search(query_vector, len(ids))
        score_by_position = {int(position): float(score) for position, score in zip(matches[0], distances[0])}
        return np.asarray([score_by_position[i] for i in range(len(ids))], dtype=float)
    return (matrix @ query_vector[0]).astype(float)


def eligibility_status(policy, intent):
    age, term = intent.get("customer_age"), intent.get("term")
    if age is not None and not (int(policy["min_age"]) <= age <= int(policy["max_age"])):
        return "ineligible"
    if term is not None and not (int(policy["min_term"]) <= term <= int(policy["max_term"])):
        return "ineligible"
    return "unknown" if age is None else "eligible"


def kg_preference(policy, intent):
    features = {
        "goal": bool(intent.get("all_goals")) and policy["financial_goal"] in intent["all_goals"],
        "risk": bool(intent.get("risk_tolerance")) and policy["risk_tolerance_inferred"] == intent["risk_tolerance"],
        "premium": bool(intent.get("premium_level")) and policy["premium_level_inferred"] == intent["premium_level"],
    }
    return sum(SMOKE_KG_PREF_WEIGHTS[name] * float(value) for name, value in features.items())


def rank_query(query, intent, alpha=SMOKE_ALPHA, beta=SMOKE_BETA, use_faiss=None):
    states = {pid: eligibility_status(policy_by_id[pid], intent) for pid in policy_ids}
    eligible = [pid for pid in policy_ids if states[pid] == "eligible"]
    if not eligible:
        return {"status": "NO_ELIGIBLE_POLICIES", "recommendations": [], "eligibility": states, "backend": BACKENDS}

    dense = minmax(dense_scores(query, eligible, use_faiss=use_faiss))
    bm25 = minmax(bm25_scores(query, eligible))
    kg_raw = np.asarray([kg_preference(policy_by_id[pid], intent) for pid in eligible], dtype=float)
    kg_norm = minmax(kg_raw)
    # This equal split exists only to exercise the smoke path; no preference weights are selected.
    bm25_kg = minmax((bm25 + kg_norm) / 2)
    candidate_scores = alpha * dense + (1 - alpha) * bm25_kg
    candidate_order = sorted(range(len(eligible)), key=lambda i: (-candidate_scores[i], eligible[i]))[:int(CONFIG["top_k_retrieval"])]
    candidate_ids = [eligible[i] for i in candidate_order]
    candidate_dense = np.asarray([dense[i] for i in candidate_order], dtype=float)
    candidate_kg = np.asarray([kg_norm[i] for i in candidate_order], dtype=float)

    # No cross-encoder is loaded in smoke mode. Dense retrieval scores are used as fallback ranking scores, never probabilities.
    ce_norm = minmax(candidate_dense)
    final_scores = beta * ce_norm + (1 - beta) * candidate_kg
    order = sorted(range(len(candidate_ids)), key=lambda i: (-final_scores[i], candidate_ids[i]))
    recommendations = [
        {
            "rank": rank,
            "policy_id": candidate_ids[index],
            "final_ranking_score": float(final_scores[index]),
            "cross_encoder_score": None,
            "cross_encoder_fallback_ranking_score": float(ce_norm[index]),
            "score_kind": "ranking_score_not_probability",
            "eligibility": states[candidate_ids[index]],
        }
        for rank, index in enumerate(order, 1)
    ]
    return {"status": "SUCCESS", "recommendations": recommendations, "eligibility": states, "backend": BACKENDS}


smoke_records = []
violations = 0
returned = 0
for query_row in queries_df.itertuples(index=False):
    intent = intents[query_row.query_id]["intent"]
    result = rank_query(query_row.raw_query, intent)
    returned += len(result["recommendations"])
    violations += sum(result["eligibility"][rec["policy_id"]] != "eligible" for rec in result["recommendations"])
    smoke_records.append({"query_id": query_row.query_id, "status": result["status"], "recommendations": result["recommendations"]})

retrieval_smoke = {
    "mode": "smoke_test_only",
    "status": "COMPLETED",
    "query_count": int(len(smoke_records)),
    "recommendation_count": int(returned),
    "eligibility_violations": int(violations),
    "eligibility_violation_rate": float(violations / returned) if returned else 0.0,
    "returned_recommendations_are_eligible": violations == 0,
    "backend": BACKENDS,
    "latency_measured": False,
    "reason": "Synthetic queries validate pipeline wiring only; no human-grade relevance metric or tuned parameter is produced.",
    "queries": smoke_records,
}
(RESULTS / "retrieval_smoke.json").write_text(json.dumps(retrieval_smoke, indent=2), encoding="utf-8")
weights_smoke = {
    "mode": "smoke_test_only",
    "status": "SKIPPED_NO_HUMAN_GRADED_LABELS",
    "selected_alpha": None,
    "selected_beta": None,
    "selected_kg_preference_weights": None,
    "sweep": None,
    "reason": "Alpha, beta, and KG preference weights require real training/validation relevance grades; no synthetic-label tuning is permitted.",
    "smoke_only_constants": {"alpha": SMOKE_ALPHA, "beta": SMOKE_BETA, "kg_preference_weights": SMOKE_KG_PREF_WEIGHTS},
}
(RESULTS / "weights_smoke.json").write_text(json.dumps(weights_smoke, indent=2), encoding="utf-8")
print(json.dumps({key: value for key, value in retrieval_smoke.items() if key != "queries"}, indent=2))
print(json.dumps(weights_smoke, indent=2))

## Assertions

In [ ]:
assert weights_smoke["status"] == "SKIPPED_NO_HUMAN_GRADED_LABELS"
assert weights_smoke["selected_alpha"] is None and weights_smoke["selected_beta"] is None
assert label_status["human_graded_rows"] == 0
assert retrieval_smoke["query_count"] == len(queries_df)
assert retrieval_smoke["eligibility_violations"] == 0
assert retrieval_smoke["eligibility_violation_rate"] == 0.0
assert all(rec["score_kind"] == "ranking_score_not_probability" for row in smoke_records for rec in row["recommendations"])
assert all(rec["eligibility"] == "eligible" for row in smoke_records for rec in row["recommendations"])

old = rank_query("age 95", {"customer_age": 95, "term": None, "all_goals": []})
assert old["status"] == "NO_ELIGIBLE_POLICIES" and not old["recommendations"]

# A constant score shift must not change rank order.
base = {"a": 0.1, "b": 0.7, "c": 0.4}
shifted = {key: value + 11 for key, value in base.items()}
assert sorted(base, key=lambda key: (-base[key], key)) == sorted(shifted, key=lambda key: (-shifted[key], key))

fallback = backend_plan(False, False, False, False)
assert fallback == {"vector": "numpy_exact", "encoder": "tfidf_fallback", "cross_encoder": "dense_ranking_score_fallback", "network": "disabled_local_only"}
without_faiss = dense_scores("life cover", [policy_ids[0], policy_ids[1]], use_faiss=False)
assert len(without_faiss) == 2 and np.isfinite(without_faiss).all()
assert minmax([1.0, 1.0]).tolist() == [0.0, 0.0]

for query_row in queries_df.itertuples(index=False):
    first = rank_query(query_row.raw_query, intents[query_row.query_id]["intent"])
    second = rank_query(query_row.raw_query, intents[query_row.query_id]["intent"])
    assert [r["policy_id"] for r in first["recommendations"]] == [r["policy_id"] for r in second["recommendations"]]
    scores = [r["final_ranking_score"] for r in first["recommendations"]]
    assert all(a >= b for a, b in zip(scores, scores[1:]))

print("Notebook 04 checks passed: strict gate, fallback paths, deterministic monotone ranking, and no synthetic-label tuning.")